# Paper Classification — Phase A + C + F (Colab T4 GPU)

Pipeline (run cells in order):
1. Setup + verify GPU
2. Clone repo + install deps
3. Upload data (Excel + .env with OPENAI_API_KEY) + sanitize
4. **Phase A** — GPT-5 panel on val + test (skip if cached parquets uploaded)
5. **Phase C** — TAPT continued MLM pretraining (~30 min)
6. Switch backbone to TAPT-adapted encoder
7. Retrain 3 tasks (train→val, single seed) with TAPT backbone (~3–4h)
8. **Phase F** — refit on train+val (~3–4h)
9. Evaluate (Phase A + Phase F + 3-way ensembles)
10. Print phase summary + download artifacts

**Total time on T4 (free tier):** ~8–10h. Within 12h session limit. Save outputs to Drive in case of disconnect.

**Cost (Phase A only):** ~$25 OpenAI for full val+test classification (one-time, cached afterward).

## 1. Verify GPU

In [ ]:
!nvidia-smi

Should show `Tesla T4`. If `command not found` → switch runtime: Runtime → Change runtime type → T4 GPU.

## 2. Clone repo + install dependencies

In [ ]:
import os
if not os.path.exists('paper-classification'):
    !git clone https://github.com/harnetlinh/paper-classification.git
%cd paper-classification
!git pull
!ls

In [ ]:
!pip install -q pyarrow openpyxl python-dotenv tenacity openai==1.57.4 transformers==4.45.2

## 3. Mount Drive (recommended) + upload data

Mounting Drive lets you save intermediate results so a Colab disconnect mid-run won't lose progress. After this cell, if you've previously run any phase, your outputs/ should appear at `/content/drive/MyDrive/btec-paper-classification/outputs/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/btec-paper-classification'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)

# Symlink local outputs/ to Drive — every save touches Drive automatically
import os, shutil
if os.path.exists('outputs') and not os.path.islink('outputs'):
    if os.listdir('outputs'):
        # Move existing local outputs into Drive (only first run)
        for item in os.listdir('outputs'):
            src = f'outputs/{item}'
            dst = f'{DRIVE_DIR}/outputs/{item}'
            if not os.path.exists(dst):
                shutil.move(src, dst)
    shutil.rmtree('outputs', ignore_errors=True)
if not os.path.islink('outputs'):
    os.symlink(f'{DRIVE_DIR}/outputs', 'outputs')
!ls -la outputs/ | head -20

In [ ]:
# Upload Excel data + .env (OPENAI_API_KEY) if not already on Drive
from google.colab import files
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx'):
    drive_data = f'{DRIVE_DIR}/data/2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx'
    if os.path.exists(drive_data):
        os.makedirs('data', exist_ok=True)
        os.symlink(drive_data, 'data/2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx')
    else:
        print('Pick the Excel data file:')
        uploaded = files.upload()
        for fn in uploaded.keys():
            os.rename(fn, f'data/{fn}')
            # Also save to Drive for next session
            os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
            shutil.copy(f'data/{fn}', f'{DRIVE_DIR}/data/{fn}')

if not os.path.exists('.env'):
    drive_env = f'{DRIVE_DIR}/.env'
    if os.path.exists(drive_env):
        shutil.copy(drive_env, '.env')
    else:
        print('Paste your .env content (OPENAI_API_KEY=sk-...). End with empty line + Ctrl+D:')
        env_content = ''
        try:
            while True:
                line = input()
                env_content += line + '\n'
        except EOFError:
            pass
        with open('.env', 'w') as f:
            f.write(env_content)
        shutil.copy('.env', drive_env)

!ls -la data/ && head -1 .env

## 4. Phase 0 — Sanitize (skip if outputs/gold_2013_2023.parquet exists)

In [ ]:
if not os.path.exists('outputs/gold_2013_2023.parquet'):
    !python sanitize.py
else:
    print('Sanitize already done. Skipping.')
    !ls outputs/*.parquet

## 5. Phase A — GPT-5 panel classification (val + test)

Cost: ~$25 OpenAI for full val (417) + test (562). Cached afterward — re-runs are FREE.

**Skip if** `outputs/llm_classify/gpt5_panel_test.parquet` and `gpt5_panel_val.parquet` already in Drive.

In [ ]:
import os.path
test_done = os.path.exists('outputs/llm_classify/gpt5_panel_test.parquet')
val_done = os.path.exists('outputs/llm_classify/gpt5_panel_val.parquet')
if not (test_done and val_done):
    !python llm_classify.py --split both
else:
    print('Phase A predictions cached. Skipping.')
!ls -la outputs/llm_classify/

## 6. Phase C — TAPT (Task-Adaptive Pretraining)

Continued MLM pretraining of SPECTER2 base on train+val+test abstracts (no labels). ~30 min on T4. Adapts encoder to corpus-specific vocabulary, helping DRIFT classes (Education economically, Non-STEM).

Output: `outputs/specter2_tapt/` — drop-in replacement for SPECTER2 base.

In [ ]:
if not os.path.exists('outputs/specter2_tapt/config.json'):
    !python tapt.py --epochs 3 --lr 5e-5 --batch-size 16
else:
    print('TAPT already done. Skipping.')
!ls outputs/specter2_tapt/ 2>/dev/null

## 7. Switch backbone to TAPT-adapted encoder

Sets `config.BACKBONE_MODEL` to point at the local TAPT directory. All subsequent training/eval/inference will use the adapted encoder.

In [ ]:
# Patch config.py to use the TAPT-adapted backbone
import os
config_text = open('config.py').read()
if 'BACKBONE_MODEL = SPECTER2_BASE' in config_text:
    new_text = config_text.replace(
        'BACKBONE_MODEL = SPECTER2_BASE',
        'BACKBONE_MODEL = str(OUTPUT_DIR / "specter2_tapt") if (OUTPUT_DIR / "specter2_tapt").exists() else SPECTER2_BASE',
    )
    open('config.py', 'w').write(new_text)
    print('Patched config.py — BACKBONE_MODEL now points at TAPT model when present.')
else:
    print('config.py already patched.')

import importlib, config
importlib.reload(config)
print(f'Active BACKBONE_MODEL: {config.BACKBONE_MODEL}')

## 8. Retrain 3 tasks (train→val) with TAPT backbone

Single-seed for time budget. Multi-seed ensemble would 3x training time and likely won't fit in the free-tier session limit alongside Phase F. Time estimate: ~3–4h total for 3 tasks at 10 epochs each on T4.

In [ ]:
!python train_specter2.py --task all

## 9. Phase F — refit on train+val

After step 8 chose hyperparams + thresholds via train→val, re-train each task on train ∪ val (2074 papers) for the SAME epoch budget. Reuses the thresholds_*.json from step 8 (no circular tuning). Saves models with `_trainval` suffix.

Time: another ~3–4h on T4.

In [ ]:
!python train_specter2.py --task all --include-val-in-train

## 10. Evaluate (Phase A + D + 3-way + final)

evaluate.py auto-picks the `_trainval` checkpoints from Phase F when they exist (loaders updated). Phase A (GPT-5 panel) and D (kNN) parquets are loaded if present and ensembled with per-class lambda.

In [ ]:
!python evaluate.py --task all

## 11. Phase summary — clean comparison table

Side-by-side: SPECTER2-only vs +GPT-5 vs +kNN vs full 3-way. Per-class breakdown for each task.

In [ ]:
!python phase_summary.py

## 12. Export human-review workbook

In [ ]:
!python export_review.py --output outputs/review.xlsx

## 13. Download artifacts (models + reports)

Models are already saved to Drive via the symlink. This zip is for local download.

In [ ]:
!zip -r artifacts.zip outputs/eval_report.json outputs/training_log_*.json outputs/thresholds_*.json outputs/review.xlsx 2>/dev/null
from google.colab import files
files.download('artifacts.zip')

## Resume after disconnect

If Colab disconnects mid-run:
1. Re-run cells 1, 2, 3 (mount drive, install deps, sanity check).
2. Skip cells whose outputs are already on Drive — they're already symlinked into `outputs/`.
3. Pick up at the first cell that still needs to run.

Each phase is idempotent / cache-aware:
- Sanitize: re-runs in <1 min if needed.
- Phase A (GPT-5): cached per-prompt — re-running is FREE.
- Phase C (TAPT): skips if `outputs/specter2_tapt/config.json` exists.
- Phase 8 (train): skips if all `model_{task}.pt` exist (manual delete to force retrain).
- Phase F: skips if all `model_{task}_trainval.pt` exist.